In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Paper reconstruction supplement worker

Frozen formal supplementary reconstruction entry. Run only after the proposed-method formal result has published its threshold and complete evaluation images, and only with explicit formal-experiment authorization.


In [ ]:
import json, os, pathlib, subprocess, sys
from google.colab import userdata

REPO='https://github.com/RICHAAARC/CEG-WM.git'
EXPECTED_EXACT='9ec454055c74cf4ed89001387c9f700e9ba5aef0'
JOB_ID='paper-main-reconstruction-v1'
MAIN_JOB_ID='paper-main-v1'
checkout=pathlib.Path('/content/cegwm-paper-reconstruction')
runtime_root=pathlib.Path('/content/cegwm-paper-runtime/reconstruction-detection')
drive_root=pathlib.Path('/content/drive/MyDrive/CEG-WM/PaperFormal-V1')
if not checkout.exists(): subprocess.run(['git','clone',REPO,str(checkout)],check=True)
subprocess.run(['git','-C',str(checkout),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(checkout),'checkout','--detach',EXPECTED_EXACT],check=True)
head=subprocess.run(['git','-C',str(checkout),'rev-parse','HEAD'],check=True,capture_output=True,text=True).stdout.strip()
dirty=subprocess.run(['git','-C',str(checkout),'status','--porcelain'],check=True,capture_output=True,text=True).stdout.strip()
assert head==EXPECTED_EXACT and not dirty
subprocess.run([sys.executable,'-m','pip','install','diffusers<0.40','transformers','accelerate'],check=True)
child_env=dict(os.environ)
child_env['PYTHONPATH']=str(checkout/'src')+os.pathsep+str(checkout)
child_env['HF_TOKEN']=userdata.get('HF_TOKEN') or ''
child_env['CEG_WM_ROOT_KEY']=userdata.get('CEG_WM_ROOT_KEY') or ''
assert child_env['HF_TOKEN'] and child_env['CEG_WM_ROOT_KEY']
command=[sys.executable,'-m','experiments.run_paper_reconstruction_worker','--job-id',JOB_ID,'--main-job-id',MAIN_JOB_ID,'--expected-exact',EXPECTED_EXACT,'--drive-root',str(drive_root),'--runtime-root',str(runtime_root)]
completed=subprocess.run(command,cwd=checkout,env=child_env,check=False)
if completed.returncode != 0:
    state_path=drive_root/'reconstruction'/JOB_ID/'job_state.json'
    state=json.loads(state_path.read_text(encoding='utf-8')) if state_path.exists() else {}
    raise RuntimeError(f"worker exited {completed.returncode}: error_code={state.get('error_code')} error={state.get('error')}")
output_root=drive_root/'reconstruction'/JOB_ID
final_path=output_root/'reconstruction_final.json'
state_path=output_root/'job_state.json'
if final_path.exists():
    public=json.loads(final_path.read_text(encoding='utf-8'))
    print({'method_id':public['method_id'],'status':public['status'],'result_package_produced':public['result_package_produced']})
else:
    state=json.loads(state_path.read_text(encoding='utf-8'))
    print({'status':state['status'],'result_package_produced':False})
